# Avaliação do Modelo

## 1. Objetivo

Este notebook apresenta a avaliação final do modelo selecionado na etapa de modelagem do projeto.

No notebook `03_modelagem.ipynb`, foram comparados diferentes algoritmos de classificação utilizando validação cruzada sobre o conjunto de treino. Entre os modelos avaliados, o **Random Forest** apresentou o melhor desempenho, considerando principalmente a PR-AUC, adotada como métrica principal devido ao desbalanceamento da variável-alvo.

Nesta etapa, o modelo selecionado será treinado utilizando o conjunto de treino e avaliado no conjunto de teste, que permaneceu reservado durante o processo de comparação e seleção dos algoritmos.

A avaliação incluirá:

- ROC-AUC e PR-AUC;
- precision, recall e F1-score;
- matriz de confusão;
- interpretação do comportamento do classificador;
- análise da importância preditiva das variáveis;
- discussão das limitações do estudo;
- possíveis melhorias;
- conclusão do projeto.

O objetivo é avaliar a capacidade de generalização do modelo e interpretar seus resultados, preservando a separação entre desenvolvimento e avaliação final.

## 2. Configuração do ambiente

Inicialmente serão carregadas as bibliotecas necessárias para reconstrução dos dados, pré-processamento, treinamento e avaliação do modelo.

Será utilizado o mesmo `RANDOM_STATE = 42` adotado nas etapas anteriores, garantindo consistência e reprodutibilidade das divisões aleatórias realizadas no projeto.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

from sklearn.inspection import permutation_importance
from sklearn.base import clone

RANDOM_STATE = 42

## 3. Reconstrução da base de avaliação

Para que este notebook seja executável de forma independente, as bases originais serão carregadas novamente e a variável-alvo será reconstruída utilizando o mesmo critério adotado nas etapas anteriores.

Será considerado `TARGET = 1` o cliente que apresentou pelo menos uma ocorrência de atraso igual ou superior a 30 dias (`STATUS` entre `1` e `5`) no histórico observado. Os demais clientes serão classificados como `TARGET = 0`.

Também serão reaplicadas as transformações determinísticas utilizadas durante a modelagem: conversão da idade e do tempo de emprego para anos, tratamento da codificação especial de `DAYS_EMPLOYED` e representação dos valores ausentes de ocupação.

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/raw")

application = pd.read_csv(DATA_DIR / "application_record.csv")
credit = pd.read_csv(DATA_DIR / "credit_record.csv")

# Clientes presentes nas duas bases
ids_comuns = set(application['ID']).intersection(set(credit['ID']))

credit_comum = credit[
    credit['ID'].isin(ids_comuns)
].copy()

# TARGET = 1 para clientes com ao menos um atraso >= 30 dias
ids_maus_pagadores = set(
    credit_comum.loc[
        credit_comum['STATUS'].isin(['1', '2', '3', '4', '5']),
        'ID'
    ]
)

target = pd.DataFrame({'ID': list(ids_comuns)})
target['TARGET'] = target['ID'].isin(ids_maus_pagadores).astype(int)

df = application.merge(
    target,
    on='ID',
    how='inner'
)

# Transformações determinísticas
df['AGE_YEARS'] = (
    -df['DAYS_BIRTH'] / 365.25
)

df['YEARS_EMPLOYED'] = np.where(
    df['DAYS_EMPLOYED'] == 365243,
    np.nan,
    -df['DAYS_EMPLOYED'] / 365.25
)

df['OCCUPATION_TYPE'] = (
    df['OCCUPATION_TYPE'].fillna('Unknown')
)

df = df.drop(
    columns=['DAYS_BIRTH', 'DAYS_EMPLOYED']
)

# Preditores e variável-alvo
X = df.drop(
    columns=['ID', 'FLAG_MOBIL', 'TARGET']
)

y = df['TARGET']

print(f'Dimensão de X: {X.shape}')
print(f'Dimensão de y: {y.shape}')

print('\nDistribuição do TARGET:')
print(y.value_counts())

## 4. Reconstrução dos conjuntos de treino e teste

Para reproduzir exatamente a estratégia utilizada durante a modelagem, será realizada novamente a divisão estratificada da base em **80% para treino e 20% para teste**, utilizando `RANDOM_STATE = 42`.

Como os dados e os parâmetros da divisão são os mesmos utilizados no notebook de modelagem, essa operação reproduz os conjuntos utilizados anteriormente.

O conjunto de treino será utilizado para ajustar o Random Forest selecionado. O conjunto de teste será utilizado exclusivamente para sua avaliação final.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'y_train: {y_train.shape}')
print(f'y_test:  {y_test.shape}')

print('\nDistribuição percentual do TARGET no teste:')
print(
    (y_test.value_counts(normalize=True) * 100)
    .round(2)
)

## 5. Reconstrução do Random Forest selecionado

Na etapa de modelagem foram avaliados quatro classificadores: Regressão Logística, Regressão Logística com balanceamento de classes, HistGradientBoosting e Random Forest.

Considerando os resultados obtidos por validação cruzada, o **Random Forest** apresentou o melhor desempenho entre os modelos avaliados, com ROC-AUC médio de aproximadamente **0,78** e PR-AUC médio de aproximadamente **0,40**.

A busca de hiperparâmetros realizada no notebook de modelagem produziu apenas uma pequena melhora na PR-AUC em relação à configuração original. Além disso, a configuração selecionada pela busca permaneceu pouco regularizada. Por esse motivo, foi mantida a configuração original do Random Forest, priorizando simplicidade e consistência metodológica.

Nesta etapa, o pipeline correspondente será reconstruído e ajustado utilizando exclusivamente o conjunto de treino.

In [ ]:
numeric_features = [
    'CNT_CHILDREN',
    'AMT_INCOME_TOTAL',
    'CNT_FAM_MEMBERS',
    'AGE_YEARS',
    'YEARS_EMPLOYED'
]

binary_features = [
    'FLAG_WORK_PHONE',
    'FLAG_PHONE',
    'FLAG_EMAIL'
]

categorical_features = [
    'CODE_GENDER',
    'FLAG_OWN_CAR',
    'FLAG_OWN_REALTY',
    'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE'
]

# Para modelos baseados em árvores não é necessária padronização
numeric_transformer_tree = Pipeline(
    steps=[
        (
            'imputer',
            SimpleImputer(
                strategy='median',
                add_indicator=True
            )
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            'onehot',
            OneHotEncoder(
                handle_unknown='ignore'
            )
        )
    ]
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_tree, numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('bin', 'passthrough', binary_features)
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor_tree),
        (
            'model',
            RandomForestClassifier(
                n_estimators=300,
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ]
)

print('Pipeline do Random Forest reconstruído com sucesso.')

## 6. Avaliação final no conjunto de teste

O Random Forest será ajustado utilizando todos os registros disponíveis no conjunto de treino. Em seguida, serão geradas previsões exclusivamente para o conjunto de teste reservado.

Como a variável-alvo é desbalanceada, a avaliação não será baseada apenas na acurácia. Serão analisadas a **ROC-AUC**, a **PR-AUC**, precision, recall e F1-score para cada classe, além da matriz de confusão.

A PR-AUC possui especial relevância neste problema por permitir avaliar o desempenho do modelo na identificação da classe minoritária — os clientes classificados como maus pagadores.

In [ ]:
# Ajuste exclusivamente no conjunto de treino
rf_pipeline.fit(X_train, y_train)

# Probabilidades e classificação no conjunto de teste
prob_test_rf = rf_pipeline.predict_proba(X_test)[:, 1]
pred_test_rf = rf_pipeline.predict(X_test)

roc_test_rf = roc_auc_score(
    y_test,
    prob_test_rf
)

pr_test_rf = average_precision_score(
    y_test,
    prob_test_rf
)

print('Random Forest — avaliação final no conjunto de teste')

print(f'\nROC-AUC: {roc_test_rf:.4f}')
print(f'PR-AUC:  {pr_test_rf:.4f}')

print('\nClassification report:')
print(
    classification_report(
        y_test,
        pred_test_rf,
        target_names=['Bom pagador', 'Mau pagador']
    )
)

print('Matriz de confusão:')
print(
    confusion_matrix(
        y_test,
        pred_test_rf
    )
)

### Interpretação dos resultados

O Random Forest apresentou **ROC-AUC de 0,8047**, indicando boa capacidade de ordenação entre clientes com diferentes níveis de risco. A **PR-AUC de 0,4244** também representa desempenho substancialmente superior à proporção da classe positiva na base de teste (aproximadamente 11,77%), indicando que o modelo possui capacidade preditiva relevante para identificação dos maus pagadores.

Entretanto, a análise das métricas por classe mostra uma diferença importante entre a capacidade de identificar bons e maus pagadores.

Utilizando o limiar padrão de classificação de 0,5, o modelo apresentou:

- **recall de 96% para bons pagadores**;
- **recall de 32% para maus pagadores**;
- **precision de 54% para maus pagadores**;
- **F1-score de 0,40 para maus pagadores**.

A matriz de confusão permite detalhar esse comportamento:

- **6.198** bons pagadores foram corretamente classificados;
- **236** bons pagadores foram classificados incorretamente como maus pagadores;
- **276** maus pagadores foram corretamente identificados;
- **582** maus pagadores foram classificados incorretamente como bons pagadores.

Portanto, embora o modelo apresente boa capacidade geral de discriminação, o limiar padrão de 0,5 favorece a classificação da classe majoritária e deixa de identificar uma parcela relevante dos maus pagadores.

A acurácia de aproximadamente 89% deve ser interpretada com cautela, pois cerca de 88% dos registros pertencem à classe de bons pagadores. Nesse contexto, métricas como PR-AUC, precision e recall da classe minoritária são mais informativas para avaliar a utilidade do classificador.

### Curvas ROC e Precision-Recall

As curvas ROC e Precision-Recall permitem avaliar o desempenho do classificador considerando diferentes limiares de decisão, em vez de analisar apenas a classificação produzida pelo limiar padrão de 0,5.

A curva ROC representa a relação entre a taxa de verdadeiros positivos e a taxa de falsos positivos. Já a curva Precision-Recall evidencia a relação entre precision e recall da classe positiva, sendo especialmente útil em problemas com classes desbalanceadas.

In [ ]:
RocCurveDisplay.from_predictions(
    y_test,
    prob_test_rf,
    name='Random Forest'
)

plt.title('Curva ROC — Random Forest')
plt.show()

In [ ]:
PrecisionRecallDisplay.from_predictions(
    y_test,
    prob_test_rf,
    name='Random Forest'
)

plt.title('Curva Precision-Recall — Random Forest')
plt.show()

### Interpretação das curvas

A curva ROC permanece acima da linha de referência ao longo da maior parte dos possíveis limiares de classificação, resultado consistente com a ROC-AUC de 0,8047. Isso indica que o modelo possui capacidade relevante de discriminar os clientes das duas classes.

A curva Precision-Recall evidencia de forma mais direta o desafio relacionado à classe minoritária. À medida que o recall dos maus pagadores aumenta, ocorre redução da precision, demonstrando o trade-off entre identificar uma parcela maior dos clientes de maior risco e aumentar a quantidade de falsos positivos.

Esse comportamento indica que o limiar padrão de 0,5 não deve ser interpretado necessariamente como o limiar ideal para uma aplicação prática. Em um cenário real de concessão de crédito, a escolha do ponto de corte deveria considerar os custos associados aos falsos positivos e falsos negativos e os objetivos da instituição.

Como esses custos não estão disponíveis na base utilizada, não será definido neste estudo um limiar de decisão ótimo.

## 7. Importância preditiva das variáveis

Além de avaliar o desempenho do classificador, é importante compreender quais variáveis apresentam maior contribuição para suas previsões.

Para isso será utilizada a técnica de **Permutation Importance**, que mede a redução no desempenho do modelo quando os valores de uma determinada variável são aleatoriamente embaralhados. Quanto maior a queda de desempenho, maior a dependência do modelo em relação àquela variável.

Para evitar utilizar o conjunto de teste durante essa etapa de interpretação, o conjunto de treino será novamente subdividido em treino interno e validação interna. Um novo Random Forest, com a mesma configuração do modelo selecionado, será ajustado exclusivamente nessa subdivisão.

A importância será calculada utilizando a **PR-AUC (average precision)** como métrica, mantendo consistência com a métrica principal adotada na comparação dos modelos.

Os resultados devem ser interpretados como **importância preditiva**, e não como evidência de relação causal entre as características dos clientes e a inadimplência.

In [ ]:
X_train_int, X_val_int, y_train_int, y_val_int = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train
)

rf_importance = clone(rf_pipeline)

rf_importance.fit(
    X_train_int,
    y_train_int
)

print(f'Treino interno:    {X_train_int.shape}')
print(f'Validação interna: {X_val_int.shape}')

In [ ]:
perm_importance = permutation_importance(
    rf_importance,
    X_val_int,
    y_val_int,
    scoring='average_precision',
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    'Variável': X_val_int.columns,
    'Importância média': perm_importance.importances_mean,
    'Desvio padrão': perm_importance.importances_std
}).sort_values(
    'Importância média',
    ascending=False
).reset_index(drop=True)

importance_df.round(4)

In [ ]:
importance_plot = importance_df.sort_values(
    'Importância média',
    ascending=True
)

plt.figure(figsize=(9, 7))

plt.barh(
    importance_plot['Variável'],
    importance_plot['Importância média'],
    xerr=importance_plot['Desvio padrão']
)

plt.xlabel('Redução média da PR-AUC')
plt.ylabel('Variável')
plt.title('Permutation Importance — Random Forest')

plt.tight_layout()
plt.show()

### Interpretação da importância das variáveis

A Permutation Importance indica que **idade (`AGE_YEARS`)**, **tempo de emprego (`YEARS_EMPLOYED`)** e **tipo de renda (`NAME_INCOME_TYPE`)** foram as variáveis com maior contribuição preditiva para o Random Forest na validação interna.

Na sequência aparecem características como posse de imóvel (`FLAG_OWN_REALTY`), ocupação (`OCCUPATION_TYPE`), renda total (`AMT_INCOME_TOTAL`) e escolaridade (`NAME_EDUCATION_TYPE`).

Os resultados complementam a análise exploratória realizada anteriormente. Na EDA, diversas variáveis apresentaram associações individuais relativamente pequenas com a variável-alvo. Entretanto, no modelo multivariado, a combinação dessas características permite obter capacidade preditiva consideravelmente superior à observada quando cada variável é analisada isoladamente.

É importante destacar que a Permutation Importance mede o quanto o desempenho preditivo do modelo depende de cada variável. Portanto, uma variável apresentar alta importância **não significa que ela seja causa da inadimplência**.

Além disso, variáveis correlacionadas podem compartilhar informação preditiva. Nesses casos, a importância de uma variável pode ser parcialmente absorvida por outra, de modo que os valores não devem ser interpretados como uma decomposição exata da contribuição individual de cada característica.

## 8. Limitações do estudo

Apesar do desempenho obtido pelo Random Forest, algumas limitações devem ser consideradas na interpretação dos resultados.

### Desbalanceamento da variável-alvo

Apenas aproximadamente 11,8% dos clientes foram classificados como maus pagadores. Esse desbalanceamento influencia o comportamento do classificador e ajuda a explicar por que métricas como a acurácia podem apresentar valores elevados mesmo quando o recall da classe minoritária permanece relativamente baixo.

### Definição da variável-alvo

Neste projeto, um cliente foi classificado como mau pagador quando apresentou pelo menos uma ocorrência de atraso igual ou superior a 30 dias (`STATUS` entre `1` e `5`) no histórico disponível. Portanto, os resultados dependem diretamente dessa definição operacional de inadimplência.

Outros critérios, como atrasos superiores a 60 ou 90 dias, poderiam produzir diferentes distribuições da variável-alvo e alterar o comportamento dos modelos.

### Estrutura temporal dos dados

As bases disponíveis não permitem estabelecer de maneira suficientemente clara uma separação temporal entre as características do cliente e o comportamento de crédito utilizado para construir a variável-alvo.

Por esse motivo, os resultados devem ser interpretados principalmente como uma classificação de perfis associados ao comportamento de crédito observado na base, e não como uma validação definitiva da capacidade de prever inadimplência futura.

Em uma aplicação real, seria recomendável utilizar informações disponíveis no momento da concessão do crédito e avaliar o desempenho sobre eventos de inadimplência ocorridos posteriormente.

### Limiar de classificação

As classificações apresentadas utilizaram o limiar padrão de 0,5. A escolha de um limiar adequado para uma aplicação real dependeria dos custos associados a falsos positivos e falsos negativos, além das políticas de risco da instituição.

### Generalização

Os resultados obtidos são específicos para a população e para as variáveis disponíveis neste conjunto de dados. Antes da utilização em outro contexto, seria necessária uma nova validação utilizando dados representativos da população em que o modelo seria aplicado.

### Interpretação das variáveis

A Permutation Importance identifica a contribuição das variáveis para o desempenho preditivo do modelo, mas não estabelece relações causais. Além disso, variáveis correlacionadas podem compartilhar informação, afetando suas importâncias individuais.

## 9. Possíveis melhorias

A partir dos resultados e das limitações identificadas, algumas extensões poderiam aprimorar o projeto:

- estruturar os dados temporalmente, utilizando informações disponíveis no momento da análise para prever eventos futuros de inadimplência;
- testar diferentes definições da variável-alvo, considerando diferentes níveis e períodos de atraso;
- definir o limiar de classificação a partir dos custos de falsos positivos e falsos negativos;
- desenvolver novas variáveis por meio de feature engineering;
- avaliar técnicas adicionais para tratamento do desbalanceamento das classes;
- testar outros algoritmos e estratégias de regularização;
- investigar formas adicionais de reduzir o overfitting observado no Random Forest;
- incorporar novas fontes de informação potencialmente relevantes para análise de risco;
- avaliar estabilidade e desempenho do modelo ao longo do tempo.

Essas melhorias poderiam aumentar tanto a capacidade preditiva quanto a adequação do modelo a uma aplicação real de análise de risco de crédito.

## 10. Conclusão

Este projeto desenvolveu um fluxo completo de ciência de dados aplicado à classificação de risco de crédito, abrangendo análise exploratória, tratamento e transformação dos dados, construção da variável-alvo, pré-processamento, treinamento, comparação e avaliação de diferentes algoritmos de classificação.

Entre os modelos avaliados, o **Random Forest apresentou o melhor desempenho na validação cruzada**, sendo selecionado para a avaliação final.

No conjunto de teste reservado, o modelo obteve:

- **ROC-AUC: 0,8047**;
- **PR-AUC: 0,4244**;
- **acurácia: aproximadamente 89%**;
- **precision de 54% para maus pagadores**;
- **recall de 32% para maus pagadores**;
- **F1-score de 0,40 para maus pagadores**.

Os resultados mostram que o modelo possui capacidade relevante de discriminação e de ordenação de risco, embora o desempenho na identificação da classe minoritária utilizando o limiar padrão de 0,5 ainda apresente espaço para melhoria.

A análise por Permutation Importance indicou **idade, tempo de emprego e tipo de renda** como as variáveis de maior contribuição preditiva para o Random Forest, seguidas por características relacionadas à posse de imóvel, ocupação, renda e escolaridade. Essas importâncias devem ser interpretadas como associações preditivas, e não como relações causais.

De forma geral, o projeto demonstra que a combinação das características disponíveis permite construir um classificador com desempenho significativamente superior ao que seria obtido pela análise isolada das variáveis. Ao mesmo tempo, as limitações relacionadas ao desbalanceamento das classes, à definição da variável-alvo, à estrutura temporal dos dados e à escolha do limiar de decisão mostram que o modelo deve ser interpretado como um estudo de classificação e análise de risco, e não como um sistema pronto para decisões automáticas de concessão de crédito.

A aplicação em um ambiente real exigiria validação temporal, definição de critérios de negócio, monitoramento contínuo e avaliação dos impactos das decisões produzidas pelo modelo.